<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Personal data in voice

**[Personal data in voice](https://learning.nextia-ai.com/courses/speech/m09/personal-data-in-voice/)** · Speech AI: Transcribe, Understand and Speak · Module 9, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find a card number that a caller said aloud, using the word times, replace it with a beep in the audio and with a label in the transcript, and write a privacy check that proves no card number is left.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | Word times and the normalisation guide, from [Word times and alignment](https://learning.nextia-ai.com/courses/speech/m05/word-times-and-alignment/) and [Normalise before you compare](https://learning.nextia-ai.com/courses/speech/m02/normalise-before-you-compare/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m09/personal-data-in-voice/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M09-L02-personal-data-in-voice/personal-data-in-voice-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 3 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/calls/phone/C-0128.wav": "0b13d91f03204848d364c6ecf7adef4c05fc11ab68da9c226e543fa0cf95213a",
    "compare.py": "99a57df2715cd87e50ce2b967afd530067dc626149f960b32d38f683d39c905f",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The problem: a card number in a recording

Larkfield records calls with consent, to train agents and check quality. Sometimes a caller reads a card number aloud, even when nobody asked. Then the recording and its transcript hold payment data, and every person and system that can open them is a risk. The card industry's rules (PCI DSS) say that card data must be protected wherever it is stored, so one card number in a recording puts the whole recording system under those rules.

The rules for Larkfield's systems: the IVR never asks for a card number by speech (keypad only, and kept out of logs); a card number in a recorded call is **removed from the transcript and beeped in the audio**; and a check proves that none is left. The [AI security course](https://learning.nextia-ai.com/courses/ai-security/) covers the law (consent, retention and deletion).

## 2. What a card number looks like

A card number has 13 to 19 digits and ends with a **check digit** (the Luhn check). The check catches most typing mistakes, and it helps us find card numbers: a random run of 16 digits passes only 1 time in 10.

The number in our calls is 4111 1111 1111 1111, a well-known test number that no bank issues. Run the next cell.

> **Predict.** Which of the three texts holds a card number?

In [ ]:
import re

def luhn_ok(number):
    """The Luhn check: card numbers (and Larkfield's order numbers) end with a check digit."""
    if not number.isdigit() or len(number) < 2:
        return False
    total = 0
    for i, ch in enumerate(reversed(number)):
        d = int(ch) * (2 if i % 2 else 1)
        total += d - 9 if d > 9 else d
    return total % 10 == 0

def card_numbers(text, language="en"):
    """Digit runs of 13 to 19 digits that pass the Luhn check, after the normalisation guide."""
    return [r for r in re.findall(r"\d+", cp.normalise(text, "full", language)) if 13 <= len(r) <= 19 and luhn_ok(r)]

for text in ["My card is four one one one, one one one one, one one one one, one one one one.",
             "The order number is 444 919.", "Call me on 4111 1111 1111 1112."]:
    print(card_numbers(text), "<-", text)

> **Check.** Only the first: the words become digits after the normalisation guide, and the 16 digits pass the check. The order number has 6 digits. The last one has 16 digits but fails the check.

## 3. Find the digits in the word times

Call C-0128 is about a double charge. The caller reads a card number aloud: a practice problem we planted. The labels have the time of every spoken word, so we can find the run of number words and its start and end time. In production, the recogniser gives the word times (Module 5).

In [ ]:
calls = load_calls()
call = calls["C-0128"]
words = [(w["w"], w["start"], w["end"]) for t in call["turns"] for w in t["words"]]
CARD = "[CARD NUMBER REMOVED]"

def redact_words(words, language="en"):
    """words: [(word, start, end)]. Returns (words with each card number replaced, the (start, end) times to beep)."""
    out, spans, i = [], [], 0
    norm = [cp.normalise(w, "full", language) for w, _, _ in words]
    while i < len(words):
        j, digits = i, ""
        while j < len(words) and (norm[j].isdigit() or (norm[j] == "" and j > i)):
            digits += norm[j]
            j += 1
        if j > i and 13 <= len(digits) <= 19 and luhn_ok(digits):
            out.append((CARD, words[i][1], words[j - 1][2]))
            spans.append((words[i][1], words[j - 1][2]))
            i = j
        else:
            out.append(words[i])
            i += 1
    return out, spans

redacted_words, spans = redact_words(words)
print("beep from", [(round(s, 2), round(e, 2)) for s, e in spans])

> **Check.** One span, from 35.01 to 37.98 seconds.

## 4. Beep the audio

The next cell replaces the span, with 0.15 s more on each side, with a 1,000 Hz tone. Listen to both players (they do not start by themselves). The voices are synthetic.

In [ ]:
def beep(x, sr, spans, freq=1000, pad=0.15):
    """Replace each span (and `pad` seconds on each side) with a tone, so nobody can hear the digits again."""
    y = x.copy()
    for s, e in spans:
        a, b = max(0, int((s - pad) * sr)), min(len(y), int((e + pad) * sr))
        y[a:b] = 0.1 * np.sin(2 * np.pi * freq * np.arange(b - a) / sr)
    return y

x, sr = read_audio("data/calls/phone/C-0128.wav")    # both sides mixed, 8 kHz
y = beep(x, sr, spans)
a, b = int(30 * sr), int(42 * sr)                     # from 30 to 42 seconds
print("before:"); display(Audio(x[a:b], rate=sr))
print("after:");  display(Audio(y[a:b], rate=sr))

The next cell draws the waveform before and after: the beep is the flat block.

In [ ]:
t = np.arange(a, b) / sr
fig, axes = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
axes[0].plot(t, x[a:b], lw=0.4); axes[0].set_title("before")
axes[1].plot(t, y[a:b], lw=0.4, color="tab:red"); axes[1].set_title("after: the card number is a tone")
for s, e in spans:
    axes[0].axvspan(s, e, color="orange", alpha=0.3)
axes[1].set_xlabel("seconds"); plt.tight_layout(); plt.show()

## 5. Redact the transcript

In [ ]:
def redact_text(text, language="en"):
    """The transcript without card numbers (written as digits or as words)."""
    red, _ = redact_words([(w, 0.0, 0.0) for w in text.split()], language)
    return " ".join(w for w, _, _ in red)

original = reference(call)
redacted = redact_text(original)
for turn in call["turns"]:
    if turn.get("pii"):
        print("before:", turn["text"]); print("after: ", redact_text(turn["text"]))
print("card numbers left in the redacted transcript:", card_numbers(redacted))

> **Check.** The caller's turn now says `[CARD NUMBER REMOVED]`, and no card number is left. The order number 444 919 stays: it has 6 digits, and Grace's team needs it.

## 6. Change one thing: a misheard digit

A recogniser makes mistakes. If it hears the last digit as "seven", the 16 digits fail the Luhn check.

> **Predict.** Does `card_numbers` still find the number?

In [ ]:
misheard = "my card number is four one one one, one one one one, one one one one, one one one seven"
print("Luhn-valid card numbers found:", card_numbers(misheard))
broad = [r for r in re.findall(r"\d+", cp.normalise(misheard)) if len(r) >= 12]
print("the broad rule (any run of 12 or more digits):", broad)

> **Check.** No: the Luhn check rejects it, but 15 of the 16 real digits are still in the text. For privacy, a miss is worse than a false alarm. The **broad rule** removes every run of 12 or more digits, whatever its check digit. Order numbers have 6 digits, so they stay.

> **On recognised transcripts.** The course team ran this on the 15 calls with a spoken card number, with Whisper `base` word times instead of the script's. The strict rule (13 to 19 digits, Luhn) found 2 of 15 card numbers: Whisper often writes a wrong digit or splits the number. The broad rule found 13, and left no card number in the text or in the beeped audio of those calls. For privacy, a miss is worse than a false alarm: use the broad rule, and send doubtful calls to a person.

## 7. Failure case: a check on the raw text

A common mistake: you search the raw transcript for 13 to 19 digits in a row.

In [ ]:
print("a regular expression on the raw text:", re.findall(r"\d{13,19}", original))
print("after the normalisation guide:       ", card_numbers(original))

> **Check.** The regular expression finds nothing, because the transcript writes the number in groups of four ("4111 1111 ..."), and a recogniser may write it in words. The check passes, and the card number stays. **Fix:** normalise first (digits joined, number words as digits), then search.

## 8. Your turn: the privacy check

> **Your turn.** Write `privacy_problems(texts, language="en")`. `texts` is a dictionary from a name to a text. Return a list of `(name, number)` pairs: one for each card number that is still in each text (13 to 19 digits that pass the Luhn check, after the normalisation guide), in the order of the dictionary. Return an empty list when every text is clean.

In [ ]:
def privacy_problems(texts, language="en"):
    ...  # your code
    return None

In [ ]:
test_texts = {
    "original": original,
    "redacted": redacted,
    "order only": "Thank you. That's order 444 919.",
    "spoken": "it's four one one one, one one one one, one one one one, one one one one",
    "two cards": "cards 4111111111111111 and 5500 0000 0000 0004",
}
problems = privacy_problems(test_texts)
print(problems)

In [ ]:
expect_hash('problems', problems, 'f1c0560cc20c50ee')

> **Check.** Your check finds 4 problems in the test texts, and none in the redacted transcript. In the course project, the same check runs on every transcript and call record before they are stored.

## Next

You found a card number with word times, beeped it, redacted the transcript, and wrote a privacy check. The [lesson page](https://learning.nextia-ai.com/courses/speech/m09/personal-data-in-voice/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Voice cloning, speaker checks and misuse](https://learning.nextia-ai.com/courses/speech/m09/voice-cloning-speaker-checks-and-misuse/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m09/personal-data-in-voice/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.